In [ ]:
import _polars_runtime_32
import _polars_runtime_32._polars_runtime as rt
import polars as pl

flags = _polars_runtime_32.BUILD_FEATURE_FLAGS
assert isinstance(flags, str)
print(f"BUILD_FEATURE_FLAGS = {flags!r}")

print(f"extension module: {rt.__name__}")
print(f"runtime version:  {rt.__version__}")
print(f"polars frontend:  {pl.__version__}")
assert rt.__version__ == pl.__version__, (rt.__version__, pl.__version__)
print("runtime and frontend versions match")

In [ ]:
# Registration: the polars frontend selects this runtime as its engine (polars._plr IS the
# extension module) unless another polars runtime takes precedence.
import importlib.util
import os
import sys

import _polars_runtime_32._polars_runtime as rt
import polars
import polars._plr

# polars picks the first runtime it can load, in the order compat, 64, 32, unless
# POLARS_FORCE_PKG / POLARS_PREFER_PKG says otherwise. With another runtime installed,
# polars may legitimately select that one instead of this package.
other_runtimes = [m for m in ("_polars_runtime_compat", "_polars_runtime_64") if importlib.util.find_spec(m)]
forced = os.environ.get("POLARS_FORCE_PKG")
preferred = os.environ.get("POLARS_PREFER_PKG")
expect_runtime_32 = forced == "32" or (forced is None and (preferred == "32" or not other_runtimes))

plr = sys.modules["polars._plr"]
print(f"registered runtime: {plr.__name__}")
if expect_runtime_32:
    assert plr.__name__ == "_polars_runtime_32._polars_runtime", plr.__name__
    assert plr is rt, "polars._plr is not the runtime-32 extension module"
    print("polars._plr is the runtime-32 module — registration OK")
else:
    print(f"Skipping the registration check: other runtimes {other_runtimes}, "
          f"POLARS_FORCE_PKG={forced!r}, POLARS_PREFER_PKG={preferred!r}")

In [ ]:
# The "32" in runtime-32: this runtime indexes rows with 32-bit integers.
import importlib.util
import os

import _polars_runtime_32._polars_runtime as rt
import polars as pl

idx = rt.get_index_type()
print(f"runtime index dtype: {idx!r}")
assert "UInt32" in str(idx), idx

# polars picks the first runtime it can load, in the order compat, 64, 32, unless
# POLARS_FORCE_PKG / POLARS_PREFER_PKG says otherwise. With another runtime installed,
# polars may legitimately select that one instead of this package.
other_runtimes = [m for m in ("_polars_runtime_compat", "_polars_runtime_64") if importlib.util.find_spec(m)]
forced = os.environ.get("POLARS_FORCE_PKG")
preferred = os.environ.get("POLARS_PREFER_PKG")
expect_runtime_32 = forced == "32" or (forced is None and (preferred == "32" or not other_runtimes))

# The frontend reports the same index type when this runtime is the one it selected.
if expect_runtime_32:
    assert pl.get_index_type() == pl.UInt32, pl.get_index_type()
    print("index type is UInt32 — the 32-bit runtime is active")
else:
    print(f"Skipping the frontend index-type check: polars may have selected one of {other_runtimes}")

In [ ]:
# Real compute through the runtime: aggregate 1M rows. This exercises the Rust
# engine's parallel + SIMD paths.
import polars as pl

print("about to run 1M-row group_by/sum")
n = 1_000_000
df = pl.DataFrame({"g": [i % 7 for i in range(n)], "v": range(n)})
out = df.group_by("g").agg(pl.col("v").sum()).sort("g")
assert out.height == 7
assert out["v"].sum() == n * (n - 1) // 2
print(out)
print(f"threads = {pl.thread_pool_size()}")
print("1M-row group_by/sum through polars-runtime-32 OK")

In [ ]:
import importlib.machinery
import os

import _polars_runtime_32._polars_runtime as rt

for cls in ("PyDataFrame", "PySeries", "PyLazyFrame", "PyExpr"):
    assert hasattr(rt, cls), f"missing {cls}"
    print(f"has {cls}")

so_path = rt.__file__
size_mib = os.path.getsize(so_path) / (1024 * 1024)
print(f"{so_path}: {size_mib:.0f} MiB")
assert so_path.endswith(tuple(importlib.machinery.EXTENSION_SUFFIXES)), so_path
assert size_mib > 10, "suspiciously small runtime library"
print("PyO3 surface + native extension module OK")